In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
# matplotlib.use('Agg')
import datetime

%matplotlib inline
from finrl.meta.preprocessor.yahoodownloader import YahooDownloader
from finrl.meta.preprocessor.preprocessors import FeatureEngineer, data_split
from finrl.meta.env_stock_trading.env_stocktrading import StockTradingEnv
from finrl.agents.stablebaselines3.models import DRLAgent
from stable_baselines3.common.logger import configure
from finrl.meta.data_processor import DataProcessor

from finrl.plot import backtest_stats, backtest_plot, get_daily_return, get_baseline
from pprint import pprint

import sys
sys.path.append("../FinRL")

import itertools

In [ ]:
from finrl import config
from finrl import config_tickers
import os
from finrl.main import check_and_make_directories
from finrl.config import (
    DATA_SAVE_DIR,
    TRAINED_MODEL_DIR,
    TENSORBOARD_LOG_DIR,
    RESULTS_DIR,
    INDICATORS,
    TRAIN_START_DATE,
    TRAIN_END_DATE,
    TEST_START_DATE,
    TEST_END_DATE,
    TRADE_START_DATE,
    TRADE_END_DATE,
)
check_and_make_directories([DATA_SAVE_DIR, TRAINED_MODEL_DIR, TENSORBOARD_LOG_DIR, RESULTS_DIR])

In [ ]:
TODAY = datetime.datetime.today().strftime('%Y-%m-%d')

In [ ]:
TRAIN_START_DATE = '2008-01-01'
TRAIN_END_DATE = '2023-01-01'
TRADE_START_DATE = '2023-01-01'
TRADE_END_DATE = TODAY

In [ ]:
ticker_list = ['AAPL', 'AMGN', 'AMZN', 'AXP', 'BA', 'CAT', 'CRM', 'CSCO', 'CVX', 'DIS', 'GS', 'HD', 'HON', 'IBM', 'JNJ', 'JPM', 'KO', 'MCD', 'MMM', 'MRK', 'MSFT', 'NKE', 'NVDA', 'PG', 'SWH', 'TRV', 'UNH', 'V', 'VZ', 'WMT']

In [ ]:
df = YahooDownloader(start_date = TRAIN_START_DATE,
                     end_date = TRADE_END_DATE,
                     ticker_list = ticker_list).fetch_data()

In [ ]:
fe = FeatureEngineer(
                    use_technical_indicator=True,
                    tech_indicator_list = INDICATORS,
                    use_vix=True,
                    use_turbulence=True,
                    user_defined_feature = False)

processed = fe.preprocess_data(df)

In [ ]:
list_ticker = processed["tic"].unique().tolist()
list_date = list(pd.date_range(processed['date'].min(),processed['date'].max()).astype(str))
combination = list(itertools.product(list_date,list_ticker))

processed_full = pd.DataFrame(combination,columns=["date","tic"]).merge(processed,on=["date","tic"],how="left")
processed_full = processed_full[processed_full['date'].isin(processed['date'])]
processed_full = processed_full.sort_values(['date','tic'])

processed_full = processed_full.fillna(0)

In [ ]:
processed_full.sort_values(['date','tic'],ignore_index=True).head(20)

In [ ]:
INDICATORS

In [ ]:
close_df = processed_full.pivot_table(index='date', columns='tic', values='close', aggfunc='mean')
close_df

In [ ]:
import reservoirpy as rpy
from reservoirpy.nodes import Reservoir, Ridge
from reservoirpy.observables import mse, nrmse, rmse
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import time

In [ ]:
def normalize_data(data):
    return (data - data.min()) / (data.max() - data.min())

In [ ]:
def calculate_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse_val = np.sqrt(mean_squared_error(y_true, y_pred))
    mask = y_true != 0
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100
    mse_score = mse(y_true, y_pred)
    rmse_score = rmse_val  # Change the variable name here
    nmrse_mean = nrmse(y_true, y_pred, norm_value=y_true.mean())
    nmrse_maxmin = nrmse(y_true, y_pred, norm_value=y_true.max() - y_true.min())
    r2 = r2_score(y_true, y_pred)
    return mae, rmse_score, mape, mse_score, rmse_score, nmrse_mean, nmrse_maxmin, r2

In [ ]:
def filter_data(dataframe, ticker, column):
    return dataframe[dataframe['tic'] == ticker][column]

In [ ]:
def train_rc_model(train, test, ticker, column, horizon):
    
    train = filter_data(train, ticker, column)
    test = filter_data(test, ticker, column)
    
    Dat = np.array(train)
    dat = np.array(test)

    # Set constants
    SEED = 42
    VERBOSE = True
    units = 10000
    z = horizon
    if __name__ == "__main__":
        rpy.set_seed(SEED)
        rpy.verbosity(0)

        if VERBOSE:
            print("Train", Dat.shape)
            print("max:", Dat.max(), "min:", Dat.min())
            print("mean:", Dat.mean(), "std:", Dat.std())

            print("Test", dat.shape)
            print("max:", dat.max(), "min:", dat.min())
            print("mean:", dat.mean(), "std:", dat.std())

        x, x1, minima, maxima = normalize_data(Dat, dat)

        n_lags = z

        X, y, X_test, y_test = create_lagged_dataset_from_full(x, x1, n_lags, z)
        
        #y = np.concatenate((x[z:], x1[:z])).reshape(-1, 1)
        #X_test = x1[:len(X1)-z].reshape(-1, 1)
        #y_test = x1[z:].reshape(-1, 1)
        #X = x.reshape(-1, 1)

        print(X.shape)
        print(y.shape)
        print(X_test.shape)
        print(y_test.shape)

        # Reservoir parameters
        units = 50
        leak_rate = 0.5
        rho = 1.0
        input_scaling = 1.0
        rc_connectivity = 0.15
        input_connectivity = 0.2
        fb_connectivity = 1.1
        regularization_coef = 1e-6
        warmup = 200
        feedback = False

        rng = np.random.default_rng(SEED)
        W = rng.random((units, units)) - 0.5
        input_bias = True  # Set to True or False depending on your choice
        Win = rng.random((units, 1 + int(input_bias))) - 0.5
        Wfb = rng.random((units, 1)) - 0.5

        mask = rng.random((units, units))
        W[mask > rc_connectivity] = 0

        mask = rng.random((units, Win.shape[1]))
        Win[mask > input_connectivity] = 0

        mask = rng.random((units, Wfb.shape[1]))
        Wfb[mask > fb_connectivity] = 0

        Win = Win * input_scaling

        # Calculate the spectral radius
        original_spectral_radius = np.max(np.abs(np.linalg.eigvals(W)))

        # Rescale the matrix to reach the requested spectral radius:
        W = W * (rho / original_spectral_radius)

        # Create a reservoir
        reservoir = Reservoir(
            units,
            lr=leak_rate,
            sr=rho,
            input_bias=input_bias,
            input_scaling=input_scaling,
            rc_connectivity=rc_connectivity,
            input_connectivity=input_connectivity,
            fb_connectivity=fb_connectivity,
        )

        readout = Ridge(ridge=regularization_coef)

        if feedback:
            reservoir = reservoir << readout

        esn = reservoir >> readout

        internal_trained = reservoir.run(X)

        reservoir.reset()

        plt.figure()
        plt.plot(internal_trained[:200, :21])
        plt.title("Activations $\\mathbf{x}(n)$ from Reservoir Neurons ID 0 to 20 for 200 time steps")
        plt.show()

        esn = esn.fit(X, y, warmup=warmup)

        y_pred = esn.run(X_test)

        mae, rmse, mape, mse_score, rmse_score, nmrse_mean, nmrse_maxmin, r2 = calculate_metrics(y_test, y_pred)

        print("Mean Absolute Error (MAE): {:.4f}".format(mae))
        print("Root Mean Squared Error (RMSE): {:.4f}".format(rmse))
        print("Mean Absolute Percentage Error (MAPE): {:.4f}%".format(mape))
        print("R-squared: {:.4f}".format(r2))
        print("\n********************")
        print(f"Errors computed over {len(X_test)} time steps")
        print("\nMean Squared error (MSE):\t%.4e" % mse_score)
        print("Root Mean Squared error (RMSE):\t%.4e" % rmse_score)
        print("Normalized RMSE (based on mean):\t%.4e" % nmrse_mean)
        print("Normalized RMSE (based on max - min):\t%.4e" % nmrse_maxmin)
        print("********************")

        plt.figure(figsize=(16, 6))
        plt.plot(y_pred, color="red", lw=1.5, label="Predictions")
        plt.plot(y_test, color="blue", lw=0.75, label="Ground truth")
        plt.title("Prediction with Reservoir Computing")
        plt.legend()
        plt.show()

        return esn      